# Experimento 03: XGBoost (Gradient Boosting Secuencial)

Entramos en la arquitectura de *Boosting*. A diferencia de Random Forest, donde los árboles se crean de forma aislada y votan al final, **XGBoost (eXtreme Gradient Boosting)** construye los árboles de manera secuencial. Cada nuevo árbol se diseña específicamente para reducir el error residual que dejaron los árboles anteriores.

En este experimento realizaremos un **Estudio de Ablación cruzado**: compararemos la efectividad de inyectar datos sintéticos (SMOTE) frente a utilizar algoritmos de penalización matemática nativa (*Cost-Sensitive Learning* / *Class Weights*).

### Paso 1: Carga MLOps y Cálculo de Penalizaciones Estadísticas
Cargamos las matrices y utilizamos la fórmula estadística `balanced` de Scikit-Learn para calcular dinámicamente un "Peso de Penalización" individual para cada ticket, inversamente proporcional a la rareza de su clase.

In [1]:
import scipy.sparse
import pandas as pd
import numpy as np
import time
from sklearn.utils.class_weight import compute_sample_weight

print("Iniciando carga de matrices en memoria...")
start_load = time.time()

# 1. CARGA INGLÉS
en_X_train = scipy.sparse.load_npz("../data/features/en_X_train_tfidf.npz")
en_X_val = scipy.sparse.load_npz("../data/features/en_X_val_tfidf.npz")
en_y_train = pd.read_csv("../data/features/en_y_train.csv").iloc[:, 0]
en_y_val = pd.read_csv("../data/features/en_y_val.csv").iloc[:, 0]

# 2. CARGA ESPAÑOL
es_X_train = scipy.sparse.load_npz("../data/features/es_X_train_tfidf.npz")
es_X_val = scipy.sparse.load_npz("../data/features/es_X_val_tfidf.npz")
es_y_train = pd.read_csv("../data/features/es_y_train.csv").iloc[:, 0]
es_y_val = pd.read_csv("../data/features/es_y_val.csv").iloc[:, 0]

# 3. CÁLCULO DE PESOS ESTADÍSTICOS (Cost-Sensitive Learning)
# Utilizamos la heurística "balanced" = Total muestras / (N_clases * Frecuencia de la clase)
pesos_en = compute_sample_weight(class_weight='balanced', y=en_y_train)
pesos_es = compute_sample_weight(class_weight='balanced', y=es_y_train)

print(f"✅ Datasets y arrays de penalización calculados en {round(time.time() - start_load, 2)} segundos.")

# 4. AUDITORÍA MATEMÁTICA: ¿Qué pesos ha decidido la fórmula?
def mostrar_pesos(y_train, pesos_calculados, idioma):
    # Juntamos etiquetas y pesos, y sacamos los valores únicos
    df_pesos = pd.DataFrame({'Clase': y_train, 'Peso_Castigo': pesos_calculados}).drop_duplicates().sort_values(by='Peso_Castigo')
    print(f"\n--- MAPA DE PENALIZACIÓN ({idioma}) ---")
    for _, row in df_pesos.iterrows():
        print(f"Clase: {row['Clase']:<35} -> Multiplicador de Error: x{round(row['Peso_Castigo'], 2)}")

mostrar_pesos(en_y_train, pesos_en, "INGLÉS")
mostrar_pesos(es_y_train, pesos_es, "ESPAÑOL")

Iniciando carga de matrices en memoria...
✅ Datasets y arrays de penalización calculados en 0.69 segundos.

--- MAPA DE PENALIZACIÓN (INGLÉS) ---
Clase: Technical Support                   -> Multiplicador de Error: x0.45
Clase: Product Support                     -> Multiplicador de Error: x0.71
Clase: Customer Service                    -> Multiplicador de Error: x0.87
Clase: IT Support                          -> Multiplicador de Error: x1.1
Clase: Billing and Payments                -> Multiplicador de Error: x1.3
Clase: Service Outages and Maintenance     -> Multiplicador de Error: x3.33
Clase: Sales and Pre-Sales                 -> Multiplicador de Error: x4.3

--- MAPA DE PENALIZACIÓN (ESPAÑOL) ---
Clase: Technical Support                   -> Multiplicador de Error: x0.45
Clase: Product Support                     -> Multiplicador de Error: x0.71
Clase: Customer Service                    -> Multiplicador de Error: x0.87
Clase: IT Support                          -> Multiplicad

### Paso 2: Experimento A (XGBoost + Pesos de Clase Estrictos)

Entrenamos el primer modelo de Gradient Boosting secuencial. En lugar de generar datos falsos con SMOTE, utilizamos la matriz original desbalanceada y le inyectamos los "Pesos de Castigo" a la función de pérdida del algoritmo (*Cost-Sensitive Learning*). 

**Nota de MLOps:** XGBoost es un algoritmo matemático estricto escrito en C++. A diferencia de Random Forest, no acepta que las etiquetas (las colas de soporte) sean cadenas de texto (Strings). Requiere obligatoriamente que las clases estén codificadas como números enteros (0, 1, 2...). Utilizaremos `LabelEncoder` para traducir los textos a números antes de entrenar, y luego revertiremos la traducción para imprimir el informe.

In [3]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
import time

# 1. CODIFICADOR DE ETIQUETAS (String -> Int -> String)
# XGBoost colapsa si le pasas texto. Necesita números del 0 al 6.
le = LabelEncoder()
en_y_train_enc = le.fit_transform(en_y_train)
en_y_val_enc = le.transform(en_y_val)

es_y_train_enc = le.transform(es_y_train) # Usamos el mismo codificador
es_y_val_enc = le.transform(es_y_val)


# 2. FUNCIÓN FÁBRICA PARA XGBOOST (Con inyección de pesos)
def train_evaluate_xgb_weights(X_train, y_train_enc, pesos, X_val, y_val_enc, label_encoder, exp_name):
    print(f"\n=======================================================")
    print(f"--- ENTRENANDO XGBOOST (PESOS NATIVOS) - {exp_name} ---")
    print("⏳ Por favor, espera. Calculando gradientes...")
    
    # INSTANCIACIÓN
    # n_estimators=100 (100 árboles secuenciales)
    # n_jobs=-1 (Usar todos los hilos del procesador)
    model = XGBClassifier(
        n_estimators=100, 
        random_state=42, 
        n_jobs=-1,
        eval_metric='mlogloss'
    )
    
    # ENTRENAMIENTO (¡AQUÍ SUCEDE LA MAGIA DE LOS PESOS!)
    start_train = time.time()
    # Le pasamos el parámetro explícito 'sample_weight' con la lista de multiplicadores
    model.fit(X_train, y_train_enc, sample_weight=pesos)
    train_time = round(time.time() - start_train, 4)
    
    # INFERENCIA
    start_inf = time.time()
    y_pred_enc = model.predict(X_val) # Devuelve números (0, 1, 2...)
    inf_time_ms = round((time.time() - start_inf) * 1000, 2)
    
    # DECODIFICACIÓN (Números -> Texto legible para el informe)
    y_pred = label_encoder.inverse_transform(y_pred_enc)
    y_val = label_encoder.inverse_transform(y_val_enc)
    
    # REPORTE
    print(f"✅ ¡Entrenamiento Finalizado!")
    print(f"[T. Entrenamiento: {train_time} seg | T. Inferencia: {inf_time_ms} ms]\n")
    print(classification_report(y_val, y_pred, zero_division=0))
    
    # RETORNO PARA EL TRACKER
    f1_macro = round(f1_score(y_val, y_pred, average='macro', zero_division=0), 4)
    report_dict = classification_report(y_val, y_pred, output_dict=True, zero_division=0)
    f1_minority = round(report_dict.get('Service Outages and Maintenance', {}).get('f1-score', 0), 4)
    
    return train_time, inf_time_ms, f1_macro, f1_minority

# 3. EJECUTAMOS EXPERIMENTO (XGBoost + Pesos, SIN SMOTE)
en_train_xgb_w, en_inf_xgb_w, en_f1_xgb_w, en_f1min_xgb_w = train_evaluate_xgb_weights(
    en_X_train, en_y_train_enc, pesos_en, en_X_val, en_y_val_enc, le, "INGLÉS"
)

es_train_xgb_w, es_inf_xgb_w, es_f1_xgb_w, es_f1min_xgb_w = train_evaluate_xgb_weights(
    es_X_train, es_y_train_enc, pesos_es, es_X_val, es_y_val_enc, le, "ESPAÑOL"
)


--- ENTRENANDO XGBOOST (PESOS NATIVOS) - INGLÉS ---
⏳ Por favor, espera. Calculando gradientes...
✅ ¡Entrenamiento Finalizado!
[T. Entrenamiento: 100.031 seg | T. Inferencia: 104.37 ms]

                                 precision    recall  f1-score   support

           Billing and Payments       0.82      0.76      0.79       381
               Customer Service       0.41      0.49      0.45       569
                     IT Support       0.43      0.43      0.43       451
                Product Support       0.48      0.44      0.46       701
            Sales and Pre-Sales       0.41      0.48      0.44       115
Service Outages and Maintenance       0.49      0.61      0.55       149
              Technical Support       0.61      0.56      0.58      1102

                       accuracy                           0.53      3468
                      macro avg       0.52      0.54      0.53      3468
                   weighted avg       0.54      0.53      0.53      3468


--- E

### Paso 3: Experimento B (XGBoost + Inyección Sintética SMOTE)

Tras demostrar que el *Cost-Sensitive Learning* fracasa en espacios vectoriales NLP tan dispersos, procedemos a evaluar a XGBoost usando la filosofía opuesta: el balanceo a nivel de datos (SMOTE). 

En lugar de castigar al algoritmo, aumentamos artificialmente el volumen de las clases minoritarias creando vectores de texto sintéticos. Se retiran los Pesos de Clase para dejar que el gradiente minimice el error de forma natural.

In [4]:
from imblearn.over_sampling import SMOTE

print("--- REGENERANDO MATRICES SINTÉTICAS (SMOTE) ---")
# Como estamos en un cuaderno nuevo, tenemos que volver a invocar a SMOTE.
# OJO: Le pasamos 'en_y_train_enc' (las etiquetas ya codificadas en números)
smote = SMOTE(random_state=42)

start_smote = time.time()
en_X_train_smote, en_y_train_enc_smote = smote.fit_resample(en_X_train, en_y_train_enc)
es_X_train_smote, es_y_train_enc_smote = smote.fit_resample(es_X_train, es_y_train_enc)
print(f"✅ Matrices expandidas con éxito en {round(time.time() - start_smote, 2)} seg.\n")


# FUNCIÓN FÁBRICA PARA XGBOOST (Con SMOTE, sin Pesos)
def train_evaluate_xgb_smote(X_train_smote, y_train_enc_smote, X_val, y_val_enc, label_encoder, exp_name):
    print(f"=======================================================")
    print(f"--- ENTRENANDO XGBOOST (SMOTE) - {exp_name} ---")
    print("⏳ Entrenando sobre matriz hipertrofiada (Puede tardar 2-3 minutos)...")
    
    # 1. INSTANCIACIÓN (Limpia, sin pesos)
    model = XGBClassifier(
        n_estimators=100, 
        random_state=42, 
        n_jobs=-1,
        eval_metric='mlogloss'
    )
    
    # 2. ENTRENAMIENTO (Aquí NO pasamos sample_weight)
    start_train = time.time()
    model.fit(X_train_smote, y_train_enc_smote)
    train_time = round(time.time() - start_train, 4)
    
    # 3. INFERENCIA
    start_inf = time.time()
    y_pred_enc = model.predict(X_val)
    inf_time_ms = round((time.time() - start_inf) * 1000, 2)
    
    # 4. DECODIFICACIÓN Y REPORTE
    y_pred = label_encoder.inverse_transform(y_pred_enc)
    y_val = label_encoder.inverse_transform(y_val_enc)
    
    print(f"✅ ¡Entrenamiento Finalizado!")
    print(f"[T. Entrenamiento: {train_time} seg | T. Inferencia: {inf_time_ms} ms]\n")
    print(classification_report(y_val, y_pred, zero_division=0))
    
    # 5. RETORNO DE VARIABLES
    f1_macro = round(f1_score(y_val, y_pred, average='macro', zero_division=0), 4)
    report_dict = classification_report(y_val, y_pred, output_dict=True, zero_division=0)
    f1_minority = round(report_dict.get('Service Outages and Maintenance', {}).get('f1-score', 0), 4)
    
    return train_time, inf_time_ms, f1_macro, f1_minority

# EJECUTAMOS EXPERIMENTO B (XGBoost + SMOTE)
en_train_xgb_sm, en_inf_xgb_sm, en_f1_xgb_sm, en_f1min_xgb_sm = train_evaluate_xgb_smote(
    en_X_train_smote, en_y_train_enc_smote, en_X_val, en_y_val_enc, le, "INGLÉS"
)

es_train_xgb_sm, es_inf_xgb_sm, es_f1_xgb_sm, es_f1min_xgb_sm = train_evaluate_xgb_smote(
    es_X_train_smote, es_y_train_enc_smote, es_X_val, es_y_val_enc, le, "ESPAÑOL"
)

--- REGENERANDO MATRICES SINTÉTICAS (SMOTE) ---
✅ Matrices expandidas con éxito en 1.08 seg.

--- ENTRENANDO XGBOOST (SMOTE) - INGLÉS ---
⏳ Entrenando sobre matriz hipertrofiada (Puede tardar 2-3 minutos)...
✅ ¡Entrenamiento Finalizado!
[T. Entrenamiento: 267.8697 seg | T. Inferencia: 81.67 ms]

                                 precision    recall  f1-score   support

           Billing and Payments       0.84      0.76      0.80       381
               Customer Service       0.44      0.37      0.41       569
                     IT Support       0.49      0.27      0.35       451
                Product Support       0.48      0.37      0.41       701
            Sales and Pre-Sales       0.55      0.36      0.43       115
Service Outages and Maintenance       0.72      0.53      0.61       149
              Technical Support       0.51      0.77      0.61      1102

                       accuracy                           0.53      3468
                      macro avg       0.58  

### Paso 4: Consolidación MLOps (Registro Central de Experimentos)

Volcamos los 4 modelos de XGBoost en los Trackers Maestros. 
Registramos explícitamente en la columna de balanceo si usamos `weights` (Cost-Sensitive Learning) o `smote`. Los resultados demuestran empíricamente que la arquitectura secuencial estándar de XGBoost (`max_depth=6`) es ineficiente computacionalmente y predictivamente para matrices NLP altamente dispersas, siendo superada ampliamente por Random Forest.

In [5]:
import pandas as pd

print("--- GUARDANDO EXPERIMENTOS XGBOOST EN EL TRACKER CENTRAL ---")

# 1. CARGAMOS LOS TRACKERS FÍSICOS
tracker_en = pd.read_csv("../data/processed/tracker_en.csv")
tracker_es = pd.read_csv("../data/processed/tracker_es.csv")

# 2. EMPAQUETAMOS (INGLÉS)
nuevos_experimentos_en = [
    {
        'exp_id': 'EN_L1_TFIDF_XGB_WEIGHTS',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'xgboost',
        'balancing': 'weights',  # Nueva etiqueta para diferenciar de 'none'
        'hyperparameters': 'n_estimators=100, class_weight=balanced',
        'train_time_sec': en_train_xgb_w,
        'inference_time_ms': en_inf_xgb_w,
        'f1_macro': en_f1_xgb_w,
        'f1_minority_class': en_f1min_xgb_w
    },
    {
        'exp_id': 'EN_L1_TFIDF_XGB_SMOTE',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'xgboost',
        'balancing': 'smote',
        'hyperparameters': 'n_estimators=100',
        'train_time_sec': en_train_xgb_sm,
        'inference_time_ms': en_inf_xgb_sm,
        'f1_macro': en_f1_xgb_sm,
        'f1_minority_class': en_f1min_xgb_sm
    }
]

# 3. EMPAQUETAMOS (ESPAÑOL)
nuevos_experimentos_es = [
    {
        'exp_id': 'ES_L1_TFIDF_XGB_WEIGHTS',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'xgboost',
        'balancing': 'weights',
        'hyperparameters': 'n_estimators=100, class_weight=balanced',
        'train_time_sec': es_train_xgb_w,
        'inference_time_ms': es_inf_xgb_w,
        'f1_macro': es_f1_xgb_w,
        'f1_minority_class': es_f1min_xgb_w
    },
    {
        'exp_id': 'ES_L1_TFIDF_XGB_SMOTE',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'xgboost',
        'balancing': 'smote',
        'hyperparameters': 'n_estimators=100',
        'train_time_sec': es_train_xgb_sm,
        'inference_time_ms': es_inf_xgb_sm,
        'f1_macro': es_f1_xgb_sm,
        'f1_minority_class': es_f1min_xgb_sm
    }
]

# 4. INYECTAMOS Y SOBREESCRIBIMOS
tracker_en = pd.concat([tracker_en, pd.DataFrame(nuevos_experimentos_en)], ignore_index=True)
tracker_en.to_csv("../data/processed/tracker_en.csv", index=False)

tracker_es = pd.concat([tracker_es, pd.DataFrame(nuevos_experimentos_es)], ignore_index=True)
tracker_es.to_csv("../data/processed/tracker_es.csv", index=False)

print("✅ Experimentos de XGBoost registrados y clausurados.")

# 5. AUDITORÍA VISUAL DEL FRACASO
print("\n--- TRACKER MAESTRO ACTUALIZADO (INGLÉS) ---")
display(tracker_en.tail(4))  # Mostramos solo las últimas 4 filas para ver RF vs XGB
print("\n--- TRACKER MAESTRO ACTUALIZADO (ESPAÑOL) ---")
display(tracker_es.tail(4))

--- GUARDANDO EXPERIMENTOS XGBOOST EN EL TRACKER CENTRAL ---
✅ Experimentos de XGBoost registrados y clausurados.

--- TRACKER MAESTRO ACTUALIZADO (INGLÉS) ---


,exp_id,target_level,vectorization,model,balancing,train_time_sec,inference_time_ms,f1_macro,f1_minority_class,hyperparameters
3,EN_L1_TFIDF_RF_NONE,queue,tfidf,random_forest,none,4.9768,53.20,0.6128,0.6891,n_estimators=100
4,EN_L1_TFIDF_RF_SMOTE,queue,tfidf,random_forest,smote,15.6008,65.79,0.6849,0.7266,n_estimators=100
5,EN_L1_TFIDF_XGB_WEIGHTS,queue,tfidf,xgboost,weights,100.0310,104.37,0.5277,0.5465,"n_estimators=100, class_weight=balanced"
6,EN_L1_TFIDF_XGB_SMOTE,queue,tfidf,xgboost,smote,267.8697,81.67,0.5171,0.6124,n_estimators=100



--- TRACKER MAESTRO ACTUALIZADO (ESPAÑOL) ---


,exp_id,target_level,vectorization,model,balancing,train_time_sec,inference_time_ms,f1_macro,f1_minority_class,hyperparameters
3,ES_L1_TFIDF_RF_NONE,queue,tfidf,random_forest,none,4.8221,62.61,0.5816,0.6121,n_estimators=100
4,ES_L1_TFIDF_RF_SMOTE,queue,tfidf,random_forest,smote,15.7321,65.97,0.6502,0.7000,n_estimators=100
5,ES_L1_TFIDF_XGB_WEIGHTS,queue,tfidf,xgboost,weights,95.1703,87.80,0.5030,0.5897,"n_estimators=100, class_weight=balanced"
6,ES_L1_TFIDF_XGB_SMOTE,queue,tfidf,xgboost,smote,234.1503,83.01,0.4941,0.5940,n_estimators=100
